# Stage 6 Materials Proxy: Calibration Template

This notebook writes the table a lab user would fill after real
calibration shots. The blank measured columns are intentional and
prevent uncalibrated threshold proxies from being presented as
material-response models.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

import bessel_twin_core as bt
from Publication_Study import publication_diagnostics as pdiag
from vbb_study import setup_study, vbb_materials
from vbb_study.publication import materials as material_schema

PATHS = setup_study.bootstrap(Path.cwd())
CSV_OUT = PATHS["csv"] / "materials"
CSV_OUT.mkdir(parents=True, exist_ok=True)


## Editable Notebook Controls

<!-- STAGE88: editable controls -->

This cell exposes the intended user-editable controls for exploratory runs. The locked stage logic below is preserved: changing these controls is for local investigation unless the notebook explicitly wires a value into a regenerated canonical output. Keep QA, caveats, and fail/marginal labels visible. For fast beam-to-sample exploration use the quicklook notebook; for publication-grade outputs use the locked stage runner.


In [ ]:
# STAGE88: visible editable controls for exploratory notebook use.
# Edit NOTEBOOK_CONTROLS below and re-run this cell to apply material
# parameter overrides for downstream analysis cells.
from vbb_study.publication import notebook_controls as nb_controls

NOTEBOOK_CONTROLS = nb_controls.make_notebook_controls(
    stage='materials',
    # ── edit these for materials analysis ───────────────────────────────────
    pulse_energy_uJ=10.0,
    threshold_fluence_J_cm2=1.0,
    pulse_count=1,
)

# Wire control parameters to named variables used by downstream cells.
_p = NOTEBOOK_CONTROLS.parameters or {}
PULSE_ENERGY_uJ = float(_p.get("pulse_energy_uJ", 10.0))
THRESHOLD_FLUENCE_J_CM2 = float(_p.get("threshold_fluence_J_cm2", 1.0))
PULSE_COUNT = int(_p.get("pulse_count", 1))

try:
    display(nb_controls.describe_controls(NOTEBOOK_CONTROLS))
except NameError:
    print(nb_controls.describe_controls(NOTEBOOK_CONTROLS).to_string(index=False))


In [ ]:
# Interactive beam quicklook — adjust sliders and click "Update plots".
# Runs a fast preview only; nothing is saved and this is independent of the
# material analysis cells below.
from dataclasses import replace
from vbb_study.publication import notebook_widgets as nbw

_ql_base = bt.default_config("fast")
_panel = nbw.interactive_quicklook(_ql_base, method='holographic', preset='fast')
display(_panel)


In [2]:
summary_path = CSV_OUT / "material_proxy_fluence_threshold_summary.csv"
if summary_path.exists():
    summary = pd.read_csv(summary_path)
else:
    summary, _cases = vbb_materials.build_shortlist_design_table(
        pdiag.DEFAULT_SHORTLIST,
        preset="fast",
        path="realistic",
    )
    summary = material_schema.ordered_material_frame(summary.to_dict("records"))
    summary.to_csv(summary_path, index=False)


## Required Measurements

A calibrated material row needs material, wavelength, pulse
duration, repetition rate, pulse count, NA or cone angle, measured
modification threshold, measured line width or depth, microscope or
etch method, and uncertainty.


In [3]:
template = vbb_materials.calibration_template_from_proxy_summary(summary)
cfg = bt.default_config("fast")
template["wavelength_nm"] = cfg.laser.wavelength_m / bt.nm
template["pulse_duration_fs"] = cfg.laser.pulse_duration_s / bt.fs
template["repetition_rate_Hz"] = cfg.laser.rep_rate_Hz
template["NA"] = cfg.objective.NA
if "gamma_slm_deg" in template:
    template["cone_angle_deg"] = template["gamma_slm_deg"]
template["calibration_status"] = "uncalibrated"
template["material_model_status"] = "planning_proxy"
template["material_response_model"] = "incubation_threshold_proxy"
template = material_schema.ordered_material_frame(template.to_dict("records"))

template_path = CSV_OUT / "material_calibration_template.csv"
template.to_csv(template_path, index=False)

display_cols = [
    "case_id",
    "material_name",
    "wavelength_nm",
    "pulse_duration_fs",
    "repetition_rate_Hz",
    "pulse_count",
    "measured_threshold_fluence_J_cm2",
    "measured_line_width_um",
    "microscope_or_etch_method",
    "measurement_uncertainty_um",
    "calibration_status",
]
display(template[display_cols])
print(template_path)


,case_id,material_name,wavelength_nm,pulse_duration_fs,repetition_rate_Hz,pulse_count,measured_threshold_fluence_J_cm2,measured_line_width_um,microscope_or_etch_method,measurement_uncertainty_um,calibration_status
0,ell0_core3_L150_calibration_template,Cr:ZnSe,1029.0,260.0,100000.0,300.0,None,None,,None,uncalibrated
1,ell3_core3_L150_calibration_template,Cr:ZnSe,1029.0,260.0,100000.0,300.0,None,None,,None,uncalibrated
2,ell5_core4_L200_calibration_template,Cr:ZnSe,1029.0,260.0,100000.0,300.0,None,None,,None,uncalibrated


C:\PhD\Code\Publication_Study\outputs\csv\materials\material_calibration_template.csv


## Calibration Boundary

Filling a measured threshold alone is not enough to claim a full
ablation, void, refractive-index-change, or weld model. Fully
calibrated rows require calibration evidence and a calibrated
response-model label.
